## Following Tutorials

### Beginners Guide
---
* **Libraries:** BeautifulSoup & Requests
* **Website:** [http://quotes.toscrape.com/](http://quotes.toscrape.com/)

In [3]:
from bs4 import BeautifulSoup as bfs
import requests

In [149]:
# Retrieve data object of website's HTML, API data, status codes, & headers:
BASE_URL = "http://quotes.toscrape.com"
page = requests.get(BASE_URL) # Returns request.Response object

# Attribute returns the HTTP response body decoded into a Unicode string:
soup = bfs(page.text, "html.parser") 

#### NOTES #1: 
*Scraping Contents:*
* **Quotes:** \<span class="text"\>
* **Author:** \<small class="author"\>
* **Tags::** \<div class="tags"\>

In [23]:
quotes = soup.find_all("span", attrs={"class" : "text"})
authors = soup.find_all("small", attrs={"class" : "author"})

for i in range(len(quotes)):
    print(f"{quotes[i].text} by {authors[i].text}")

“The world as we have created it is a process of our thinking. It cannot be changed without changing our thinking.” by Albert Einstein
“It is our choices, Harry, that show what we truly are, far more than our abilities.” by J.K. Rowling
“There are only two ways to live your life. One is as though nothing is a miracle. The other is as though everything is a miracle.” by Albert Einstein
“The person, be it gentleman or lady, who has not pleasure in a good novel, must be intolerably stupid.” by Jane Austen
“Imperfection is beauty, madness is genius and it's better to be absolutely ridiculous than absolutely boring.” by Marilyn Monroe
“Try not to become a man of success. Rather become a man of value.” by Albert Einstein
“It is better to be hated for what you are than to be loved for what you are not.” by André Gide
“I have not failed. I've just found 10,000 ways that won't work.” by Thomas A. Edison
“A woman is like a tea bag; you never know how strong it is until it's in hot water.” by Ele

#### NOTES #2:  
*Error*
```bash
DeprecationWarning: Call to deprecated method findAll. (Replaced by find_all) -- Deprecated since version 4.0.0. quotes = soup.findAll("span", attrs={"class" : "text"})
```

### Requests & BeautifulSoup Practice #1
---
1. **Identify the Top 10 Tags:** Locate "Top Ten Tags" section on the right side of the page.
    * Write the logic to isolate & extract these tag names.
    * Make sure to avoid tags nested underneath the individual quotes.


#### NOTES:
* `.descendants`, `.name`, `.parents`, `.self_and_parents`
* `find_all()` cannot use attribute `.descendants` 

In [48]:
top_10_tags_div = soup.find_all("div", attrs={"class" : "col-md-4 tags-box"})

In [ ]:
for parent in top_10_tags_div:
    # Test:
    # print(f"Parent Tag: {parent.text}")
    # continue 
    for child in parent.descendants:
        if child.name is not None and child.name == 'a':
            print(f"Child Tag: {child.text}")

Child Tag: love
Child Tag: inspirational
Child Tag: life
Child Tag: humor
Child Tag: books
Child Tag: reading
Child Tag: friendship
Child Tag: friends
Child Tag: truth
Child Tag: simile


2. **Extract Specific Tags per Quote:** For each quote, isolate the list of associated tags.
    * Create a system that captures each quote alongside its specific tags.
    * i.e: {*tag* : *quote 1* by *author*}

In [127]:
quotes = soup.find_all("span", attrs = {"class" : "text"})
tags_per_quote = soup.find_all("div", attrs = {"class" : "tags"})

for i in range(len(quotes)):
    print(f"Quote: {quotes[i].text}")
    for child in tags_per_quote[i]:
        if child.name is not None and child.name == 'a':
            print(f"\t{child.text}")

Quote: “The world as we have created it is a process of our thinking. It cannot be changed without changing our thinking.”
	change
	deep-thoughts
	thinking
	world
Quote: “It is our choices, Harry, that show what we truly are, far more than our abilities.”
	abilities
	choices
Quote: “There are only two ways to live your life. One is as though nothing is a miracle. The other is as though everything is a miracle.”
	inspirational
	life
	live
	miracle
	miracles
Quote: “The person, be it gentleman or lady, who has not pleasure in a good novel, must be intolerably stupid.”
	aliteracy
	books
	classic
	humor
Quote: “Imperfection is beauty, madness is genius and it's better to be absolutely ridiculous than absolutely boring.”
	be-yourself
	inspirational
Quote: “Try not to become a man of success. Rather become a man of value.”
	adulthood
	success
	value
Quote: “It is better to be hated for what you are than to be loved for what you are not.”
	life
	love
Quote: “I have not failed. I've just found

3. **Follow the Author's Link:** Identify the `(about)` link next to any author's name. 
    * Map out the logic to find this URL,
    * reques the new page, and
    * extract the author's DoB & LoB.

In [165]:
link_element = soup.find_all("a", string="(about)")

for get_about in link_element:
    NEXT_URL = get_about.get("href")
    FULL_URL = requests.compat.urljoin(BASE_URL, NEXT_URL)
    print(f"Navigating to:{FULL_URL}")
    next_page = requests.get(FULL_URL) # Returns request.Response object
    next_soup = bfs(next_page.text, "html.parser") 
    get_DoB = next_soup.find("span", attrs={"class":"author-born-date"})
    get_LoB = next_soup.find("span", attrs={"class":"author-born-location"})
    print(f"\tBirth: {get_DoB.text}")
    print(f"\tLocation: {get_LoB.text}")


Navigating to:http://quotes.toscrape.com/author/Albert-Einstein
	Birth: March 14, 1879
	Location: in Ulm, Germany
Navigating to:http://quotes.toscrape.com/author/J-K-Rowling
	Birth: July 31, 1965
	Location: in Yate, South Gloucestershire, England, The United Kingdom
Navigating to:http://quotes.toscrape.com/author/Albert-Einstein
	Birth: March 14, 1879
	Location: in Ulm, Germany
Navigating to:http://quotes.toscrape.com/author/Jane-Austen
	Birth: December 16, 1775
	Location: in Steventon Rectory, Hampshire, The United Kingdom
Navigating to:http://quotes.toscrape.com/author/Marilyn-Monroe
	Birth: June 01, 1926
	Location: in The United States
Navigating to:http://quotes.toscrape.com/author/Albert-Einstein
	Birth: March 14, 1879
	Location: in Ulm, Germany
Navigating to:http://quotes.toscrape.com/author/Andre-Gide
	Birth: November 22, 1869
	Location: in Paris, France
Navigating to:http://quotes.toscrape.com/author/Thomas-A-Edison
	Birth: February 11, 1847
	Location: in Milan, Ohio, The Unite

4. **Scrape Multiple Pages:** Notice the "Next" button.
    * Build the logic using loops & URL concatenation to visit all 10 pages, and
    * print the total number of unique authors on the site

In [246]:

def unique_authors_count(good_soup):
    ls = []
    authors = good_soup.find_all("small", attrs={"class" : "author"})
    for author in authors:
        ls.append(author.text)
    return len(list(set(ls)))
    print(len(ls))
    print(len(list(set(ls))))

In [191]:
unique_authors_count(soup)

10
8


In [328]:
def append_url(base_url, next_pg):
    out = requests.compat.urljoin(base_url, next_pg)
    return out
# next_page = requests.get(FULL_URL) # Returns request.Response object

In [ ]:
def new_soup(url):
 next_page = requests.get(url)
 soup_base = bfs(next_page.text, "html.parser") 
 base_page = soup_base.find("li", attrs={"class":"next"})
 print("NEW SOOP", base_page)
 
 return base_page, soup_base 

url = "http://quotes.toscrape.com/"
(base_page, new_soup_base) = new_soup(url)

# print(base_page.a.get("href"))
# # while base_page is not None:
dic = {}
for i in range(3):
    get_page_url = base_page.a.get("href") 
    page_num = "Page " + str(int(get_page_url[-2]) - 1)
    dic[page_num] =  unique_authors_count(new_soup_base)
    new_url = append_url(url, get_page_url)
    print("NEW URL", new_url)
    (base_page, new_soup_base) = new_soup(new_url)
print(dic)



NEW SOOP <li class="next">
<a href="/page/2/">Next <span aria-hidden="true">→</span></a>
</li>
NEW URL http://quotes.toscrape.com/page/2/
NEW SOOP <li class="next">
<a href="/page/3/">Next <span aria-hidden="true">→</span></a>
</li>
NEW URL http://quotes.toscrape.com/page/3/
NEW SOOP <li class="next">
<a href="/page/4/">Next <span aria-hidden="true">→</span></a>
</li>
NEW URL http://quotes.toscrape.com/page/4/
NEW SOOP <li class="next">
<a href="/page/5/">Next <span aria-hidden="true">→</span></a>
</li>
{'Page 1': 8, 'Page 2': 10, 'Page 3': 9}


## Following Documentation

### Requests
---

`.get()` Method

In [10]:
r = requests.get('https://api.github.com/user', auth=('user', 'pass'))
print(f"Status Code: {r.status_code}")
print(f"Headers (content-type): {r.headers['content-type']}")
print(f"Encoding: {r.encoding}")
print(f"Text: {r.text}")
print(f"JSON: {r.json()}")

Status Code: 401
Headers (content-type): application/json; charset=utf-8
Encoding: utf-8
Text: {"message":"Requires authentication","documentation_url":"https://docs.github.com/rest/users/users#get-the-authenticated-user","status":"401"}
JSON: {'message': 'Requires authentication', 'documentation_url': 'https://docs.github.com/rest/users/users#get-the-authenticated-user', 'status': '401'}


In [ ]:
r.encoding

In [21]:
import sys
import os

# Use current working directory 
sys.path.insert(0, os.getcwd())
sys.path.append(os.path.abspath('..'))

from src.scraper import(
    _clean_base,
    _fetch_with_requests,
    _incentive_score,
    _extract_internal_links,
    _fetch_raw_html,
    scrape_venue_pages,
    INCENTIVE_PATHS,
)


In [22]:
"""
Hard Coded URL:
"""
URL = "https://26beach.com/"
BASE = _clean_base(URL)

print("BASE:", BASE)

BASE: https://26beach.com


In [25]:
"""
Functions: _fetch_with_requests(1) & _incentive_score(1)

Output: 
- Incentive Path (Incentive Type)
- Chars (Text)
- Score
- Preview
"""
for path in INCENTIVE_PATHS:
    full = BASE + path
    text = _fetch_with_requests(full)
    if(len(text) != 0):
        print("PATH:", path or "/")
        print("CHARS:", len(text))
        print("SCORE:", _incentive_score(text))
        print("PREVIEW:", text[:500])


PATH: /menu
CHARS: 2779
SCORE: 4
PREVIEW: Get Ready to Treat Yourself Whether you’re celebrating something special or just looking for a fancy night out, we’ve got you covered. So sit back, relax, and get ready to treat yo’ self to some seriously indulgent eats! BRUNCH DINNER COCKTAILS BURGERS Get Ready to Treat Yourself Whether you’re celebrating something special or just looking for a fancy night out, we’ve got you covered. So sit back, relax, and get ready to treat yo’ self to some seriously indulgent eats! Get Ready to Treat Yoursel
PATH: /
CHARS: 1256
SCORE: 3
PREVIEW: CONTACT 3100 Washington Blvd, Venice, CA

90292, United States. Monday to Thursday 10am-9pm Friday to Sunday 9am-9pm (310) 823-7526 Yelp Instagram Facebook Copyright © 2010-2026 All rights reserved. OTHER LINKS Yelp Instagram Facebook Copyright © 2010-2026 All rights reserved. CONTACT 3100 Washington Blvd, Venice, CA

90292, United States. Monday to Thursday 10am-9pm Friday to Sunday 9am-9pm (310) 823-7526 Yelp Inst

In [26]:
"""
Functions: _fetch_raw_html(1), _extract_internal_links(2), scrape_venue_pages(1)

Output:

"""
html = _fetch_raw_html(BASE)
links = _extract_internal_links(html, BASE)
print("DISCOVERED LINKS:")
for link, score in links[:10]:
    print(score, link)
final = scrape_venue_pages(URL)
print("\nFINAL TEXT:")
print(final[:2000])

DISCOVERED LINKS:
1 https://26beach.com/menu
1 https://26beach.com/monthly-events
Scraping: https://26beach.com/happy-hour
Scraping: https://26beach.com/happyhour
Scraping: https://26beach.com/specials
Scraping: https://26beach.com/deals
Scraping: https://26beach.com/entertainment
Scraping: https://26beach.com/live-music
Scraping: https://26beach.com/events
Scraping: https://26beach.com/menu
Scraping: https://26beach.com
  -> JS renderer for 4 sparse path(s)...

FINAL TEXT:
Get Ready to Treat Yourself Whether you’re celebrating something special or just looking for a fancy night out, we’ve got you covered. So sit back, relax, and get ready to treat yo’ self to some seriously indulgent eats! BRUNCH DINNER COCKTAILS BURGERS Get Ready to Treat Yourself Whether you’re celebrating something special or just looking for a fancy night out, we’ve got you covered. So sit back, relax, and get ready to treat yo’ self to some seriously indulgent eats! Get Ready to Treat Yourself Whether you’re cele